# Joining and Analyzing Customers and Orders 

In [14]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("test").master("local[*]").getOrCreate()
spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [15]:
spark = SparkSession.builder.appName("APP1").getOrCreate()
# read and show data 
df_customers = spark.read.option("header",True).csv("work/customers.csv")

In [18]:
df_customers.show(5)

+-----------+----------+---------+-----------+-------+-----------------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|
+-----------+----------+---------+-----------+-------+-----------------+---------+
|          0|Customer_0|     Pune|West Bengal|  India|       2023-10-10|     True|
|          1|Customer_1|Bangalore|    Gujarat|  India|       2023-10-19|    False|
|          2|Customer_2|Bangalore|  Karnataka|  India|       2023-02-10|     True|
|          3|Customer_3|Bangalore|  Telangana|  India|       2023-03-24|     True|
|          4|Customer_4|Hyderabad|  Telangana|  India|       2023-06-04|    False|
+-----------+----------+---------+-----------+-------+-----------------+---------+
only showing top 5 rows



In [19]:
df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- is_active: string (nullable = true)



In [20]:
df_orders = spark.read.option("header",True).csv("work/orders.csv")

In [21]:
df_orders.show(5)

+--------+-----------+----------+-----------------+---------+
|order_id|customer_id|order_date|     total_amount|   status|
+--------+-----------+----------+-----------------+---------+
|       0|      54138|2024-10-09|813.1420169946641|Delivered|
|       1|      45056|2024-01-14|948.2041026445735|  Shipped|
|       2|       3758|2024-12-25|859.3306972195478|  Pending|
|       3|      57481|2024-02-25|48.74859397050256|Cancelled|
|       4|       2823|2024-12-23|266.6808800664503|  Shipped|
+--------+-----------+----------+-----------------+---------+
only showing top 5 rows



In [22]:
df_orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- total_amount: string (nullable = true)
 |-- status: string (nullable = true)



In [23]:
# Analysis on Orders 

In [24]:
# lets do join between two tables 

In [48]:
custmers_orders_df = df_customers.join(df_orders,on="customer_id" ,how="inner")

In [49]:
custmers_orders_df.count()

169673

In [50]:
custmers_orders_df.show(5)

+-----------+----------+---------+-----------+-------+-----------------+---------+--------+----------+-----------------+---------+
|customer_id|      name|     city|      state|country|registration_date|is_active|order_id|order_date|     total_amount|   status|
+-----------+----------+---------+-----------+-------+-----------------+---------+--------+----------+-----------------+---------+
|          0|Customer_0|     Pune|West Bengal|  India|       2023-10-10|     True|    2552|2024-06-20|947.8474347297807|  Shipped|
|          1|Customer_1|Bangalore|    Gujarat|  India|       2023-10-19|    False|  144006|2024-09-01|945.1783810475038|Cancelled|
|          2|Customer_2|Bangalore|  Karnataka|  India|       2023-02-10|     True|  136245|2024-01-18|995.5799727041127|Cancelled|
|          5|Customer_5|Hyderabad|West Bengal|  India|       2023-07-26|     True|   75022|2024-01-04| 768.167470468336|  Shipped|
|          6|Customer_6|Hyderabad|  Karnataka|  India|       2023-08-07|    False| 

In [82]:
# Total Oreder per customer
from pyspark.sql import functions as F 
customer_total_spend =custmers_orders_df.groupby("customer_id","name").agg(F.count("order_id").alias("Total")).show()

+-----------+---------------+-----+
|customer_id|           name|Total|
+-----------+---------------+-----+
|      89354| Customer_89354|    2|
|     164686|Customer_164686|    1|
|      62526| Customer_62526|    3|
|         75|    Customer_75|    2|
|     149345|Customer_149345|    1|
|     157519|Customer_157519|    1|
|      76390| Customer_76390|    2|
|       3899|  Customer_3899|    1|
|     114208|Customer_114208|    1|
|     104521|Customer_104521|    2|
|     128130|Customer_128130|    3|
|      32728| Customer_32728|    4|
|      94702| Customer_94702|    1|
|       8062|  Customer_8062|    2|
|     168291|Customer_168291|    1|
|     102380|Customer_102380|    2|
|       3877|  Customer_3877|    1|
|      29393| Customer_29393|    4|
|     142757|Customer_142757|    2|
|     124262|Customer_124262|    1|
+-----------+---------------+-----+
only showing top 20 rows



In [83]:
# Total Spend per customer 
customer_total_spend = custmers_orders_df.groupby("customer_id","name").agg(F.sum("total_amount").alias("Total Amount")).orderBy(col("Total Amount").desc()).show()

+-----------+---------------+------------------+
|customer_id|           name|      Total Amount|
+-----------+---------------+------------------+
|       4680|  Customer_4680| 5138.631065963802|
|     144160|Customer_144160| 4590.783328014365|
|       1822|  Customer_1822| 4587.463046775093|
|     119990|Customer_119990| 4584.205837473942|
|     132046|Customer_132046| 4373.860921890816|
|     116459|Customer_116459| 4296.914086806486|
|      19141| Customer_19141| 4244.870486493495|
|      72794| Customer_72794| 4188.187293249394|
|       3255|  Customer_3255|  4183.03741171044|
|     150927|Customer_150927| 4163.655455957863|
|      61531| Customer_61531| 4163.381512985349|
|      27584| Customer_27584| 4157.363677138945|
|     122760|Customer_122760| 4103.596837084002|
|     160283|Customer_160283|4081.5235548997844|
|      72820| Customer_72820|4056.0094736610854|
|     134539|Customer_134539| 4050.669958257735|
|      73396| Customer_73396|4025.7348357282794|
|      96399| Custom

In [55]:
# Avrege Spend per Customer
customer_avg_spend =custmers_orders_df.groupby('customer_id','name').agg(avg("total_amount").alias("Avredge")).orderBy(col("Avredge").desc())

In [56]:
customer_avg_spend.show()

+-----------+---------------+-----------------+
|customer_id|           name|          Avredge|
+-----------+---------------+-----------------+
|     108333|Customer_108333|999.9944314947996|
|     163792|Customer_163792|  999.97538759395|
|      59970| Customer_59970|999.9230808355982|
|     154751|Customer_154751|999.9160853903466|
|     133876|Customer_133876|999.8863893638942|
|       9919|  Customer_9919|999.8657977955661|
|     105671|Customer_105671|999.8453263969482|
|      74614| Customer_74614|999.8393651902434|
|      77334| Customer_77334|999.8377068244474|
|     135281|Customer_135281|999.8320507356099|
|     130387|Customer_130387|999.8166760468108|
|      74109| Customer_74109|999.7824653073789|
|      50133| Customer_50133| 999.751961454337|
|       3275|  Customer_3275|999.7487061537723|
|      62529| Customer_62529|999.7465451798445|
|     112833|Customer_112833|999.7435904731503|
|      19896| Customer_19896|999.7374964864836|
|      12827| Customer_12827|999.7355224

In [58]:
#Order By status
orders_status_count = custmers_orders_df.groupBy('status').count()
orders_status_count.show()

+---------+-----+
|   status|count|
+---------+-----+
|  Shipped|42395|
|Cancelled|42606|
|Delivered|42626|
|  Pending|42046|
+---------+-----+



In [65]:
# Order By month
order_month = custmers_orders_df.groupby(month(col("order_date")).alias("Month")).count().orderBy('Month')
order_month.show()

+-----+-----+
|Month|count|
+-----+-----+
|    1|14406|
|    2|13553|
|    3|14591|
|    4|13858|
|    5|14233|
|    6|13990|
|    7|14380|
|    8|14401|
|    9|14094|
|   10|14317|
|   11|14019|
|   12|13831|
+-----+-----+



In [76]:
# rank the customers by the total spent
from pyspark.sql import functions as F 

from pyspark.sql.window import Window
windowSpec = Window.partitionBy("customer_id").orderBy("total_amount")
custmers_orders_df.withColumn("Total Amount", F.sum("total_amount").over(windowSpec))\
.withColumn('dense_rank',dense_rank().over(windowSpec)).show()


+-----------+---------------+---------+-----------+-------+-----------------+---------+--------+----------+------------------+---------+------------------+----------+
|customer_id|           name|     city|      state|country|registration_date|is_active|order_id|order_date|      total_amount|   status|      Total Amount|dense_rank|
+-----------+---------------+---------+-----------+-------+-----------------+---------+--------+----------+------------------+---------+------------------+----------+
|     100014|Customer_100014|  Kolkata| Tamil Nadu|  India|       2023-09-09|     True|   93436|2024-02-01| 44.11971284812507|Delivered| 44.11971284812507|         1|
|     100014|Customer_100014|  Kolkata| Tamil Nadu|  India|       2023-09-09|     True|  119351|2024-11-21|511.77561575851945|Delivered| 555.8953286066445|         2|
|     100090|Customer_100090|Bangalore| Tamil Nadu|  India|       2023-10-16|    False|   32747|2024-08-07| 467.6907169389581|  Pending| 467.6907169389581|         1

In [85]:
# Finding customers with the High order Frequency but Low Total Spend 
customer_avg_spend.printSchema() , customer_total_spend.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- Avredge: double (nullable = true)



AttributeError: 'NoneType' object has no attribute 'printSchema'

In [86]:
customer_spend_vs_orders = customer_avg_spend.join(orders_status_count ,)